# {{TITLE}}

Incident analysis — {{DATE_LONG}}

**Impact**: _describe scope and severity_

**Timeline**: _when detected, when resolved_

**Root cause**: _TBD — fill in after investigation_

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.home() / '.claude/skills/jupyter-notebook/lib'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
import re
import json

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

from jupyter_utils import get_db_engine, query_df, setup_plotting
plt, sns = setup_plotting()

engine = get_db_engine()
print('Ready for incident analysis')

## Log Ingestion

Load relevant logs — Docker logs, application logs, or database audit trail.

In [ ]:
# Option 1: Parse Docker logs from a file
# Capture logs first: docker logs <container> --since 2h > /tmp/incident_logs.txt
log_file = '/tmp/incident_logs.txt'

def parse_log_lines(filepath):
    """Parse log lines into a DataFrame with timestamp, level, message."""
    records = []
    ts_pattern = re.compile(
        r'(\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2})'
    )
    level_pattern = re.compile(
        r'\b(DEBUG|INFO|WARNING|WARN|ERROR|CRITICAL|FATAL)\b'
    )
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ts_match = ts_pattern.search(line)
            lvl_match = level_pattern.search(line)
            records.append({
                'timestamp': pd.to_datetime(ts_match.group(1)) if ts_match else pd.NaT,
                'level': lvl_match.group(1) if lvl_match else 'UNKNOWN',
                'message': line,
            })
    return pd.DataFrame(records)

# Uncomment when log file is ready:
# logs = parse_log_lines(log_file)
# print(f'Loaded {len(logs)} log lines')
# logs.head()

In [ ]:
# Option 2: Query audit trail from ESP database
# Adjust the time window and table name for your incident

# audit_logs = query_df("""
#     SELECT created_at, action, entity_type, entity_id, user_id, details
#     FROM audit_log
#     WHERE created_at >= NOW() - INTERVAL '4 hours'
#     ORDER BY created_at DESC
#     LIMIT 500
# """)
# print(f'Found {len(audit_logs)} audit entries')
# audit_logs.head()

## Event Timeline

Visualize the sequence of events leading to and during the incident.

In [ ]:
# Timeline visualization — uncomment once logs are loaded

# def plot_event_timeline(df, time_col='timestamp', level_col='level'):
#     """Plot log events on a timeline, colored by severity."""
#     df = df.dropna(subset=[time_col]).copy()
#     if df.empty:
#         print('No timestamped events to plot')
#         return
#
#     level_colors = {
#         'DEBUG': '#95a5a6', 'INFO': '#3498db',
#         'WARNING': '#f39c12', 'WARN': '#f39c12',
#         'ERROR': '#e74c3c', 'CRITICAL': '#8e44ad',
#         'FATAL': '#8e44ad', 'UNKNOWN': '#bdc3c7'
#     }
#
#     # Events per minute by level
#     df['minute'] = df[time_col].dt.floor('min')
#     grouped = df.groupby(['minute', level_col]).size().unstack(fill_value=0)
#
#     fig, ax = plt.subplots(figsize=(16, 6))
#     colors = [level_colors.get(c, '#bdc3c7') for c in grouped.columns]
#     grouped.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.9)
#     ax.set_title('Events per Minute by Severity', fontweight='bold')
#     ax.set_ylabel('Event Count')
#     ax.set_xlabel('Time')
#     ax.legend(title='Level')
#     plt.xticks(rotation=45)
#     plt.tight_layout()
#     plt.show()
#
# plot_event_timeline(logs)

## Error Analysis

Drill into errors — frequency, patterns, stack traces.

In [ ]:
# Error frequency and patterns
# errors = logs[logs['level'].isin(['ERROR', 'CRITICAL', 'FATAL'])].copy()
#
# if not errors.empty:
#     # Extract first line of each error as the error type
#     errors['error_type'] = errors['message'].str[:100]
#     error_counts = errors['error_type'].value_counts().head(10)
#
#     fig, ax = plt.subplots(figsize=(14, 6))
#     error_counts.plot(kind='barh', ax=ax, color='#e74c3c')
#     ax.set_title('Top 10 Error Patterns', fontweight='bold')
#     ax.set_xlabel('Occurrences')
#     ax.invert_yaxis()
#     plt.tight_layout()
#     plt.show()
#
#     print(f'\nTotal errors: {len(errors)}')
#     print(f'Unique patterns: {errors["error_type"].nunique()}')
# else:
#     print('No errors found in logs')

## Database Investigation

Check database state around the incident window.

In [ ]:
# Active connections and locks during incident
# connections = query_df("""
#     SELECT state, COUNT(*) as count,
#            MAX(EXTRACT(EPOCH FROM (now() - state_change))) as max_age_sec
#     FROM pg_stat_activity
#     WHERE datname = current_database()
#     GROUP BY state
#     ORDER BY count DESC
# """)
# connections

## Findings

### Root Cause
_What caused the incident?_

### Contributing Factors
- _Factor 1_
- _Factor 2_

### Impact
- **Duration**: _start to resolution_
- **Users affected**: _count or scope_
- **Data impact**: _any data loss or corruption_

### Action Items
- [ ] _Immediate fix_
- [ ] _Preventive measure_
- [ ] _Monitoring improvement_